# Parameter transfer for drug-selection QAOA

**Main question:** Can angles trained on one drug portfolio be reused on related portfolios **without retraining**, or with only **limited retraining**, while retaining useful solution quality? Does this reduce quantum resource requirements relative to independently training QAOA—and is it competitive with classical knapsack?

- Uses finite-shot measurements **during training and evaluation**. The simulator internally computes statevectors, but the optimization and transfer policy do **not** inspect exact amplitudes/probabilities.
- Compares **unchanged angles**, **gamma scale-adjusted angles** (only for approximately proportional cost Hamiltonians), **short retraining**, and **independent full retraining**.
- Exact classical optimum is used **only afterward for benchmarking**; it is not passed to the trainer or to the transfer rule.
- Local simulator timing **is not QPU speed**. Report circuit executions and transpiled gates as hardware *resource estimates*, not proof of quantum advantage.

All experiments use the same reusable `qaoa_utils.py` and `classical_utils.py`.

In [ ]:
%pip install -q "qiskit>=2.4,<3" "qiskit-addon-opt-mapper>=0.1,<0.2" numpy pandas scipy matplotlib

## 1. Configure the experiment

The first `N_DRUGS` rows form the source instance. We train it **once**. Variants perturb one factor at a time (value, cost, budget, roster replacement, or size). Set `INCLUDE_LARGER=True` only after the small tests work; extra qubits make statevector simulation expensive.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from classical_utils import dynamic_programming, greedy, portfolio, timed_solver
from qaoa_utils import make_instance, build_qaoa, train_qaoa, sample_qaoa, circuit_resources
from transfer_utils import transfer_cases, coefficient_gamma_transfer

CSV_PATH = "Dataset.csv"
N_DRUGS = 5
BUDGET = None                # None -> 40% of total selected cost, rounded down
LAYERS = 1
SOURCE_EVALS = 60
FRESH_EVALS = 60
SHORT_EVALS = 12
TRAIN_SHOTS = 100             # shots for EVERY optimizer evaluation
FINAL_SHOTS = 250             # fixed, same for every method and instance
INCLUDE_LARGER = False        # try only after main study succeeds
MAX_SIM_QUBITS = 18
SEED = 42
TARGET_QUALITY = 0.95

raw = pd.read_csv(CSV_PATH)
raw.columns = raw.columns.str.strip()
selected = raw.head(N_DRUGS).copy()
default_budget = int(np.floor(0.4 * selected["Test Cost ($M)"].sum()))
budget = default_budget if BUDGET is None else BUDGET
source_instance = make_instance(selected, budget)
cases = transfer_cases(raw, source_instance, include_larger=INCLUDE_LARGER)
print(f"Baseline: {N_DRUGS} drugs, ${budget}M budget")
print("Variants:", [name for name, _ in cases])

## 2. Train the source problem once

This is the one-time cost that must be **amortized** across future problems. The trainer only receives finite-shot energy estimates; it cannot access the classical optimum or the simulator's exact state probabilities.

In [ ]:
source_model = build_qaoa(source_instance, layers=LAYERS, qubit_limit=MAX_SIM_QUBITS)
source_fit = train_qaoa(
    source_model,
    maxiter=SOURCE_EVALS,
    shots_per_eval=TRAIN_SHOTS,
    seed=SEED,
)
source_angles = source_fit["params"]
print("Source angles:", source_angles)
print("Source training evaluations:", source_fit["evaluations"])
print("Source training circuit executions:", source_fit["training_shots"])
print("Source Ising scale:", source_model["scale"])
print("Angle order:", [str(parameter) for parameter in source_model["parameters"]])

## 3. Compare transfer methods on each new instance

Strategies: **direct** (0 retraining evaluations), **scale-corrected direct** (if suitable), **short warm-start retraining** (with and without correction), and **independent full retraining**. Every method gets the same final shot budget. The gamma correction solves a least-squares coefficient ratio for comparable **normalized** Hamiltonians; it is exact only if the Hamiltonians are proportional up to an identity term. We explicitly skip it if the qubit count differs or proportionality is poor. We do **not** estimate correction from the mean of coefficients.

The independent full-retrain baseline uses a random start; this is a pilot with one optimizer seed, not a statistical claim about training reliability.

In [ ]:
rows = []
resource_rows = []

for case_id, (case_name, instance) in enumerate(cases):
    model = build_qaoa(instance, layers=LAYERS, qubit_limit=MAX_SIM_QUBITS)
    costs = instance["costs"]
    values = instance["values"]
    budget = instance["budget"]

    # Classical solutions provide reference scores, never training inputs.
    optimum_selection, classical_ms = timed_solver(
        dynamic_programming, costs, values, budget, repeats=5
    )
    optimum = portfolio(optimum_selection, costs, values)["value_m"]
    greedy_selection = greedy(costs, values, budget)
    greedy_value = portfolio(greedy_selection, costs, values)["value_m"]

    corrected, correction = coefficient_gamma_transfer(source_model, model, source_angles)
    # Each candidate specifies its starting angles, evaluation budget, and seed.
    retraining_seed = SEED + 100 + case_id
    candidates = [
        ("direct", source_angles, 0, None),
        ("short retrain", source_angles, SHORT_EVALS, retraining_seed),
    ]
    if correction["applied"]:
        candidates.extend([
            ("gamma corrected", corrected, 0, None),
            ("corrected + short retrain", corrected, SHORT_EVALS, retraining_seed),
        ])
    candidates.append(("fresh full retrain", None, FRESH_EVALS, SEED + 200 + case_id))

    circuit_stats = circuit_resources(model, topology="line")
    resource_rows.append({
        "Case": case_name,
        "Qubits": circuit_stats["qubits"],
        "Transpiled 2Q gates": circuit_stats["transpiled_two_qubit"],
        "Circuit depth": circuit_stats["transpiled_depth"],
        "Gamma correction applied": correction["applied"],
        "Gamma ratio": correction["ratio"],
        "Proportionality residual": correction["residual"],
    })

    for strategy, starting_angles, budget_evals, init_seed in candidates:
        if budget_evals:
            fit = train_qaoa(
                model,
                maxiter=budget_evals,
                initial_params=starting_angles,
                shots_per_eval=TRAIN_SHOTS,
                seed=init_seed,
            )
            angles = fit["params"]
            train_shots = fit["training_shots"]
            elapsed_train = fit["training_seconds"]
            evaluations = fit["evaluations"]
        else:
            angles = starting_angles
            train_shots = 0
            elapsed_train = 0.0
            evaluations = 0

        # Use the same final-shot budget and sampling seed for every strategy.
        measured = sample_qaoa(
            model,
            angles,
            shots=FINAL_SHOTS,
            optimum=optimum,
            target_fraction=TARGET_QUALITY,
            seed=SEED + case_id,
        )
        best_selection = measured["best_sampled"]
        chosen_drugs = (
            [instance["names"][index] for index in best_selection]
            if best_selection is not None else []
        )
        rows.append({
            "Case": case_name,
            "Strategy": strategy,
            "Drugs": len(costs),
            "Budget ($M)": budget,
            "Classical optimum ($M)": optimum,
            "Greedy ($M)": greedy_value,
            "Classical DP (ms)": classical_ms,
            "Best sampled ($M)": measured["best_sampled_value"],
            "Best quality": measured["best_sampled_value"] / optimum if optimum else np.nan,
            "Target hit rate": measured["prob_target_sampled"],
            "Feasible rate": measured["prob_feasible_sampled"],
            "Target hits": measured["target_hits"],
            "Train evaluations": evaluations,
            "Training shots": train_shots,
            "Final shots": FINAL_SHOTS,
            "Total local shots": train_shots + FINAL_SHOTS,
            "Local training (s)": elapsed_train,
            "Local sampling (s)": measured["sampling_seconds"],
            "Chosen drugs": chosen_drugs,
        })
    print("Finished:", case_name)

results = pd.DataFrame(rows)
resources = pd.DataFrame(resource_rows)
display_columns = [
    "Case", "Strategy", "Best quality", "Target hit rate",
    "Train evaluations", "Total local shots",
]
display(results[display_columns].round(3))

## 4. See whether training can be eliminated or reduced

The primary metric is the **fraction of final shots** yielding a feasible portfolio worth at least 95% of that instance's exact optimum. This optimum is only used by the analyst after measurement. A practical deployment would need an independently specified *business acceptance threshold*, since the optimum is not known beforehand.

A direct-transfer result that works on one variation isn't conclusive; inspect failures and robustness across different scenario types.

In [ ]:
quality = results.pivot(index="Case", columns="Strategy", values="Best quality")
hits = results.pivot(index="Case", columns="Strategy", values="Target hit rate")
quality_percent = 100 * quality
hit_percent = 100 * hits
display(quality_percent.round(1).rename_axis(columns="Best sampled value / optimum (%)"))
display(hit_percent.round(1).rename_axis(columns="Target-quality probability estimate (%)"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
quality_percent.plot(kind="bar", ax=axes[0], title="Best feasible sampled quality")
axes[0].axhline(95, linestyle="--", color="black", linewidth=1)
axes[0].set_ylabel("% of exact classical optimum")
hit_percent.plot(kind="bar", ax=axes[1], title="Target-quality hit rate")
axes[1].set_ylabel("% of finite shots")
for axis in axes:
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=65)
plt.show()

## 5. Does transfer reduce **total** resources?

Reusing angles reduces quantum training *relative to fresh QAOA*, but not necessarily versus classical dynamic programming. The source training is counted **once** in transfer strategies, not magically free. The table below aggregates the subsequent changed instances (control excluded). The shot counts are a reproducible *quantum circuit execution budget*, not predicted physical QPU time. Circuits may differ across instances in transpiled gate count.

**Caution:** Some gamma corrections are unavailable (different qubit widths or non-proportional Hamiltonians), so only compare strategies on the same set of instances. The aggregate focuses on `direct`, `short retrain`, and `fresh full retrain`, which apply to every case.

In [ ]:
# Compare methods available for every changed instance; exclude the control.
changed = results[results["Case"] != "unchanged/control"]
main_methods = ["direct", "short retrain", "fresh full retrain"]
main_results = changed[changed["Strategy"].isin(main_methods)]
summary = main_results.groupby("Strategy").agg(
    Instances=("Case", "nunique"),
    Mean_quality=("Best quality", "mean"),
    Mean_target_hit_rate=("Target hit rate", "mean"),
    Training_shots=("Training shots", "sum"),
    Final_shots=("Final shots", "sum"),
    Classical_DP_ms=("Classical DP (ms)", "sum"),
)
# Transfer methods share one source-training cost across all changed instances.
summary["One-time source training"] = [
    source_fit["training_shots"] if name != "fresh full retrain" else 0
    for name in summary.index
]
summary["Total shots incl. source"] = (
    summary["Training_shots"]
    + summary["Final_shots"]
    + summary["One-time source training"]
)
display(summary.round(3))
display(resources.round(3))

classical_seconds = changed.drop_duplicates("Case")["Classical DP (ms)"].sum() / 1000
print(
    "For QAOA-versus-classical: classical DP seconds (all changed cases):",
    round(classical_seconds, 6),
)
print("QPU runtime/fidelity advantage: NOT established from local simulation.")

## 6. Interpretation checklist

- **Transfer works** if new-instance quality/hit rates stay close to fresh full QAOA *without additional training*. Compare outcomes, not just lower training counts.
- **Short retraining helps** if it gains substantial target hit rate for noticeably fewer shots/evaluations than full training.
- **Gamma correction helps** only if its supported cases outperform the *same* uncorrected cases. The exact gamma scaling property is for proportional Hamiltonians, and our normalized operator already cancels pure uniform scaling; not every change to costs, values, budget or roster is proportional.
- **Classical advantage remains the default hypothesis:** dynamic programming solves these small integer-budget instances very cheaply. Lower training cost compared with fresh QAOA is *not* by itself a speedup over DP. Hardware speedups require real device timing, errors, latency and success rates, plus the one-time training cost amortized over how many problems are solved.
- **Statistical reliability:** 250 final shots and one training seed are a pilot. To support a robust claim, repeat on different seeds and independent held-out roster changes, report confidence intervals, and avoid selecting transfer rules based on the same held-out cases.
- **Quantum hardware compatibility:** these training rules use finite-shot bitstrings. The installed local backend is an *ideal statevector simulator* internally, so measured local runtime isn't QPU runtime. Replacing its sampling backend with a real QPU is future work.

The existing `drug_selection_benchmark.ipynb` is preserved for the single-instance baseline; this notebook adds transfer rather than replacing that benchmark.